# Pose-Controlled Image Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tigermorning/pose-image-tool/blob/main/pose_tool.ipynb)

Generate images that follow **both** a reference pose and a text prompt.

**Pipeline:** reference photo -> OpenPose keypoint detection -> skeleton image -> ControlNet -> SDXL -> output image.

| | |
|---|---|
| Base model | `stabilityai/stable-diffusion-xl-base-1.0` |
| ControlNet | `thibaud/controlnet-openpose-sdxl-1.0` |
| Pose annotator | `controlnet_aux.OpenposeDetector` (`lllyasviel/Annotators`) |
| GPU | Colab free **T4** is enough (fp16, ~10 GB peak) |
| Runtime | ~5 min setup + ~50 s per 832x1216 image |

**Before you start:** `Runtime > Change runtime type > T4 GPU`, and have 2-3 photos of people in clearly different poses ready to upload (section 3).

**Sections:** 1 Install dependencies - 2 Load models - 3 Extract pose - 4 Generate image - 5 Experiments - 6 Save results - 7 Findings

---
## 1. Install dependencies

In [ ]:
# Install the diffusion stack and the OpenPose annotator.
# controlnet_aux is pinned to 0.0.10: the older 0.0.9 constrains timm so tightly that pip cannot
# resolve it against a current timm, while 0.0.10 imports cleanly on timm 1.x.
!pip install -q "diffusers>=0.31" "transformers>=4.44" "accelerate>=0.34" safetensors
!pip install -q "controlnet_aux==0.0.10"
!pip install -q mediapipe  # only used by the optional fallback extractor in section 3b

In [ ]:
# Confirm a CUDA GPU is attached and report its VRAM before loading ~10 GB of weights.
import platform
import torch

assert torch.cuda.is_available(), 'No GPU attached. Colab: Runtime > Change runtime type > T4 GPU.'
_props = torch.cuda.get_device_properties(0)
print(f'python : {platform.python_version()}')
print(f'torch  : {torch.__version__}')
print(f'gpu    : {_props.name} ({_props.total_memory / 1024 ** 3:.1f} GB)')

# T4 has no usable bf16 path, so fp16 is used for every module in this notebook.
DTYPE = torch.float16

---
## 2. Load models

Two independent model groups are loaded here:

1. **OpenPose annotator** - detects human keypoints. It only *reads* images, it generates nothing.
2. **SDXL + ControlNet** - generates images. ControlNet is the adapter that injects the skeleton into every denoising step.

In [ ]:
# Load the OpenPose annotator (body + hand + face keypoint models from lllyasviel/Annotators).
from controlnet_aux import OpenposeDetector

openpose = OpenposeDetector.from_pretrained('lllyasviel/Annotators')
print('OpenPose annotator ready')

In [ ]:
# Load SDXL base + the OpenPose ControlNet for SDXL.
# The fp16-fix VAE is mandatory: the stock SDXL VAE overflows in fp16 and returns black images.
from diffusers import AutoencoderKL, ControlNetModel, StableDiffusionXLControlNetPipeline

BASE_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
CONTROLNET_ID = 'thibaud/controlnet-openpose-sdxl-1.0'

controlnet = ControlNetModel.from_pretrained(CONTROLNET_ID, torch_dtype=DTYPE)
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=DTYPE)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    BASE_ID,
    controlnet=controlnet,
    vae=vae,
    torch_dtype=DTYPE,
    variant='fp16',
    use_safetensors=True,
)
# Fit the pipeline to the available VRAM. A 16 GB T4 takes the fast full-GPU path; smaller
# cards fall back to sequential CPU offload (~6 GB peak, roughly 2x slower).
VRAM_GB = _props.total_memory / 1024 ** 3
if VRAM_GB >= 12:
    pipe.to('cuda')
    mode = 'full GPU'
else:
    pipe.enable_model_cpu_offload()
    mode = 'cpu offload'
pipe.enable_vae_slicing()  # decodes the latent in slices: saves VRAM, no quality cost
pipe.set_progress_bar_config(leave=False)
print(f'pipeline ready ({mode}, {VRAM_GB:.1f} GB VRAM)')

---
## 3. Extract pose

Upload the reference photos, then convert each one into a skeleton image.

- **Experiment A** (same pose, different prompts) needs **1** reference.
- **Experiment B** (same prompt, different poses) needs **2+** references with visibly different poses.

Good references: one person, full body or at least head-to-knee, limbs not overlapping the torso. Crowds and heavy occlusion are where OpenPose fails.

In [ ]:
# Upload the reference photos (jpg / png / webp). They are kept in ./references,
# and every artifact this notebook produces goes to ./samples.
from pathlib import Path

from PIL import Image, ImageOps

REF_DIR = Path('references')
OUT_DIR = Path('samples')
REF_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files

    for fname, data in files.upload().items():
        (REF_DIR / fname).write_bytes(data)
except ImportError:
    print('Not running on Colab - copy your images into ./references by hand, then re-run this cell.')

SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}
REFS = sorted(p for p in REF_DIR.iterdir() if p.suffix.lower() in SUFFIXES)
assert REFS, 'No reference images found in ./references'
print(f'{len(REFS)} reference image(s):', [p.name for p in REFS])

In [ ]:
# Helpers. The ControlNet hint and the generated image must have identical dimensions,
# so every reference is cropped to one fixed portrait resolution up front.
WIDTH, HEIGHT = 832, 1216  # SDXL-native portrait ratio; lighter on a T4 than 1024x1024


def load_ref(path, width=WIDTH, height=HEIGHT):
    """Open a reference photo, apply its EXIF rotation, and center-crop it to the target ratio."""
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    return ImageOps.fit(img, (width, height), method=Image.LANCZOS)


def extract_pose(image, include_hand=True, include_face=False, detect_resolution=512):
    """Run OpenPose on a photo and return the skeleton image used as the ControlNet hint.

    controlnet_aux resizes internally, so the result is scaled back to the input size.
    Hands are on by default (they help), faces off (they mostly add noise at this resolution).
    """
    skeleton = openpose(
        image,
        detect_resolution=detect_resolution,
        image_resolution=min(image.size),
        include_body=True,
        include_hand=include_hand,
        include_face=include_face,
    )
    return skeleton.resize(image.size, Image.LANCZOS)

In [ ]:
# Extract one skeleton per reference and save it as the repository's pose_NN.png artifact.
poses = []
for i, path in enumerate(REFS, start=1):
    ref = load_ref(path)
    skeleton = extract_pose(ref)
    skeleton.save(OUT_DIR / f'pose_{i:02d}.png')
    poses.append({'id': f'{i:02d}', 'source': path.name, 'ref': ref, 'pose': skeleton})

print('saved:', [f"pose_{p['id']}.png" for p in poses])

In [ ]:
# Inspect every reference next to its skeleton. Do this before generating anything:
# a missing arm or a merged pair of legs here will be a missing arm in the output too.
import matplotlib.pyplot as plt
import numpy as np


def show_grid(images, titles, cols=None, size=3.0):
    """Display images in a labelled grid (used for every preview and experiment below)."""
    cols = cols or len(images)
    rows = -(-len(images) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * 1.45 * rows), squeeze=False)
    flat = axes.ravel()
    for ax, img, title in zip(flat, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
    for ax in flat:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


show_grid(
    [im for p in poses for im in (p['ref'], p['pose'])],
    [t for p in poses for t in (p['source'], f"pose_{p['id']}.png")],
    cols=min(4, 2 * len(poses)),
)

### 3b. Optional fallback extractor

Skip this cell if section 3 worked.

`controlnet_aux` is lightly maintained and its imports break whenever `timm` or `huggingface_hub` shift. If the annotator refuses to load, this cell rebuilds an equivalent hint from **MediaPipe Pose** landmarks, remapped to the COCO-18 keypoint order and drawn with the standard OpenPose limb colours - which is what the ControlNet was actually trained to read. Then use `extract_pose_mediapipe` in place of `extract_pose` above.

Trade-off: MediaPipe gives body keypoints only (no hands, no face) and is less accurate on strongly foreshortened limbs.

In [ ]:
# Fallback pose extractor: MediaPipe landmarks -> OpenPose-style COCO-18 skeleton.
import cv2
import numpy as np

# MediaPipe landmark index -> COCO-18 index. Neck (COCO 1) has no MediaPipe equivalent
# and is derived from the shoulder midpoint below.
_MP_TO_COCO18 = {0: 0, 2: 15, 5: 14, 7: 17, 8: 16, 11: 5, 12: 2, 13: 6, 14: 3,
                 15: 7, 16: 4, 23: 11, 24: 8, 25: 12, 26: 9, 27: 13, 28: 10}
_LIMBS = [(1, 2), (1, 5), (2, 3), (3, 4), (5, 6), (6, 7), (1, 8), (8, 9), (9, 10),
          (1, 11), (11, 12), (12, 13), (1, 0), (0, 14), (14, 16), (0, 15), (15, 17)]
_COLORS = [(255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0), (85, 255, 0),
           (0, 255, 0), (0, 255, 85), (0, 255, 170), (0, 255, 255), (0, 170, 255), (0, 85, 255),
           (0, 0, 255), (85, 0, 255), (170, 0, 255), (255, 0, 255), (255, 0, 170), (255, 0, 85)]


def extract_pose_mediapipe(image, min_visibility=0.4):
    """Drop-in replacement for extract_pose() that does not depend on controlnet_aux."""
    import mediapipe as mp

    width, height = image.size
    with mp.solutions.pose.Pose(static_image_mode=True, model_complexity=2) as detector:
        result = detector.process(np.array(image))
    if not result.pose_landmarks:
        raise RuntimeError('MediaPipe found no person in this image.')

    points = [None] * 18
    for mp_index, coco_index in _MP_TO_COCO18.items():
        landmark = result.pose_landmarks.landmark[mp_index]
        if landmark.visibility >= min_visibility:
            points[coco_index] = (int(landmark.x * width), int(landmark.y * height))
    if points[2] and points[5]:  # neck = midpoint between the two shoulders
        points[1] = ((points[2][0] + points[5][0]) // 2, (points[2][1] + points[5][1]) // 2)

    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    for i, (a, b) in enumerate(_LIMBS):
        if points[a] and points[b]:
            cv2.line(canvas, points[a], points[b], _COLORS[i], 4)
    for i, point in enumerate(points):
        if point:
            cv2.circle(canvas, point, 4, _COLORS[i], -1)
    return Image.fromarray(canvas)

---
## 4. Generate image

One helper does all generation. Every experiment below only changes its arguments, so the variable under test is always explicit.

Key knobs:

| Argument | Effect |
|---|---|
| `conditioning_scale` | How hard the skeleton is enforced. Low = prompt wins, high = pose wins but anatomy stiffens. |
| `guidance_scale` | How hard the text prompt is enforced. |
| `seed` | Fixed by default so experiments stay single-variable. |

`conditioning_scale` is exposed as an argument so it can be swept independently; this run holds it at the 0.8 default.

In [ ]:
# Single entry point for generation: conditions on the pose skeleton and the text prompt together.
NEGATIVE = 'lowres, blurry, deformed hands, extra limbs, extra fingers, watermark, text, jpeg artifacts'


def generate(pose_image, prompt, seed=1234, steps=28, guidance_scale=6.0,
             conditioning_scale=0.8, negative_prompt=NEGATIVE):
    """Return one image that follows both `pose_image` (via ControlNet) and `prompt` (via SDXL).

    The seed is an explicit argument so any two calls can be compared as a controlled experiment.
    """
    generator = torch.Generator('cuda').manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=pose_image,
        width=pose_image.width,
        height=pose_image.height,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=conditioning_scale,
        generator=generator,
    ).images[0]

---
## 5. Experiments

Two runs, each isolating one variable. Prompts, seeds and settings are also recorded in `prompts.md`.

### Experiment A - same pose, different prompts

Pose, seed, steps, guidance and conditioning scale are all held fixed. Only the prompt string changes, so every difference in the output is attributable to the prompt. The three prompts deliberately span a *subject* change (astronaut / knight) and a *medium* change (watercolour), because those stress the pose constraint differently.

In [ ]:
# EXPERIMENT A - one pose, several prompts, identical seed and sampler settings.
# The two prompts differ in subject, wardrobe, setting and lighting, so any difference in
# the output is attributable to the prompt alone.
POSE_A = poses[0]['pose']
PROMPTS_A = [
    'A beautiful funky African woman wearing a flowing bohemian dress, seated in a chic pose, under the night sky, wet asphalt reflecting colourful neon lights, cinematic lighting, professional photography, ultra-realistic, highly detailed face',
    'A Middle Eastern warrior clad in glowing neon armour, sitting in a futuristic laundromat, studio lighting, ultra-realistic, highly detailed',
]

exp_a = []
for i, prompt in enumerate(PROMPTS_A, start=1):
    image = generate(POSE_A, prompt, seed=1234)
    image.save(OUT_DIR / f'exp_a_{i}.png')
    exp_a.append(image)

labels = [f'A{i}' for i in range(1, len(exp_a) + 1)]
show_grid([POSE_A, *exp_a], [f"pose_{poses[0]['id']}.png (fixed)", *labels])

### Experiment B - same prompt, different poses

The inverse test. One prompt and one seed, applied to every uploaded skeleton. Differences in the output isolate what the pose hint actually controls (limb placement, body orientation, framing) versus what it leaves to the prompt (identity, clothing, lighting, background).

In [ ]:
# EXPERIMENT B - one prompt, every uploaded pose, identical seed and sampler settings.
assert len(poses) >= 2, 'Upload at least two references with different poses to run experiment B.'
PROMPT_B = 'A Middle Eastern warrior clad in glowing neon armour, sitting in a futuristic laundromat, studio lighting, ultra-realistic, highly detailed'

exp_b = []
for entry in poses[:3]:
    image = generate(entry['pose'], PROMPT_B, seed=777)
    image.save(OUT_DIR / f"exp_b_{entry['id']}.png")
    exp_b.append(image)

used = poses[:len(exp_b)]
show_grid(
    [entry['pose'] for entry in used] + exp_b,
    [f"pose_{entry['id']}" for entry in used] + [f"B on pose_{entry['id']}" for entry in used],
    cols=len(used),
)

---
## 6. Save results

Writes the four filenames the repository expects, then zips `samples/` for download so the images can be committed.

In [ ]:
# Copy the canonical repository artifacts, then package everything for download.
# pose_01/pose_02 were already written in section 3.
import shutil

shutil.copy(OUT_DIR / 'exp_a_1.png', OUT_DIR / 'output_01.png')  # pose_01 + prompt A1 (astronaut)
shutil.copy(OUT_DIR / f"exp_b_{poses[1]['id']}.png", OUT_DIR / 'output_02.png')  # pose_02 + prompt B

shutil.make_archive('samples', 'zip', root_dir='.', base_dir='samples')  # cross-platform, no shell
print('samples/:', sorted(p.name for p in OUT_DIR.iterdir()))

try:
    from google.colab import files as colab_files

    colab_files.download('samples.zip')
except ImportError:
    print('samples.zip written next to the notebook')

---
## 7. Findings

Measured on the run committed to this repository: 4 images at 832x1216, 28 steps, guidance 6.0,
`controlnet_conditioning_scale` 0.8, fp16, on an 8 GB RTX 3060 Ti using `enable_model_cpu_offload()`
(torch 2.6.0+cu124, diffusers 0.39.0). Pose extraction took 5.4 s per image; generation took
435-553 s per image, mean 506 s. On a 16 GB T4 taking the full-GPU path, expect roughly 40-60 s
per image instead - the offload path trades about an order of magnitude of speed for fitting in 8 GB.

### What was changed

| Experiment | Held fixed | Varied |
|---|---|---|
| **A** | pose hint `pose_01`, seed 1234, 28 steps, guidance 6.0, conditioning 0.8 | prompt: African woman in a bohemian dress on neon-lit wet asphalt / Middle Eastern warrior in neon armour in a laundromat |
| **B** | prompt (neon-armour warrior), seed 777, all sampler settings | pose hint: `pose_01` (clean detection) / `pose_02` (tangled detection) |

Both references were seated figures. `pose_01` came out clean - all four limbs traced to the
extremities. `pose_02`, a crouch with the arms wrapped around the shins, came out tangled: the
lower-leg segments cross and extend outside the body outline, so left and right are swapped.

### What changed in the output

**A - same pose, different prompts.** The skeleton held completely. Across two unrelated subjects
the raised right hand at the head, the elbow angle, the left hand braced on a surface to the
right, the extended left leg, the bent right leg, the head tilt, and the figure's position and
scale in frame were all reproduced. Everything the skeleton does not encode moved freely: subject,
wardrobe, setting, palette, and lighting (magenta-and-orange street neon versus bright even
interior light). This is the clean half of the claim - **the skeleton owns where the body is, the
prompt owns what the body is.**

**B - same prompt, different poses.** The inverse held for the good hint and broke for the bad one.
Identity carried across both images: armour, neon trim, bearded figure, neon-lit interior. `B1`
(`pose_01`) reproduced the seated leaning pose faithfully, full body in frame. `B2` (`pose_02`)
lost the pose entirely - the crouch was replaced by a generic symmetric seated figure with both
hands on the knees, and the crop tightened from full-body to waist-up so the feet and the extended
leg disappeared.

The important part is *how* it failed. A tangled hint did not produce a tangled pose; it produced
a **default** pose. When the skeleton is incoherent, ControlNet cannot enforce anything and the
model falls back on its own prior, which is the blandest posture consistent with the prompt.
Practically: a bad hint does not corrupt pose control, it silently switches pose control off.

Two more effects worth recording:

- **The hint drives framing, not just limb placement.** `pose_01` spans most of the canvas and
  produced full-body compositions in all three images that used it. `pose_02` occupies a compact,
  confusing region and produced a much closer crop. Framing is therefore not an independent knob.
- **Prompt adherence is uneven across prompt clauses.** `neon armour` was rendered emphatically in
  every image, while `futuristic laundromat` survived only as vague panels and a counter. The model
  spent its capacity on the subject and treated the setting as optional. `studio lighting` in the
  same prompt was also overridden by neon spill from the armour.

### Limitations observed

1. **Pose detection is the ceiling, and it fails silently.** `pose_02` is the direct evidence: the
   detector produced a confident-looking but wrong skeleton, and the only visible symptom
   downstream was a pose that ignored the reference. Nothing in the generation step reports that
   the hint was bad. Inspect the section 3 preview every time.
2. **Self-occlusion is the specific trigger.** Both references were seated. The one where limbs
   wrap across each other (arms around shins) is the one that failed. Crossed and overlapping
   limbs, not seated poses in general, are the problem.
3. **The hint is 2D.** A skeleton carries no depth, so limb crossings cannot be disambiguated -
   which is the mechanism behind failure 1 and 2.
4. **Hands were malformed in all 4 images.** Fingers merged on every raised hand, and in `B2` the
   hand and the knee armour fused. Hand keypoints were enabled and `deformed hands` was in the
   negative prompt; both reduced the severity without fixing it. This is the most reliable defect
   in the pipeline.
5. **Framing cannot be set independently of the pose.** See above - crop follows the skeleton's
   extent in frame.
6. **Setting and lighting clauses lose to subject clauses.** If the background matters, it needs
   its own emphasis, and competing light sources ("studio lighting" plus "glowing neon") will not
   both be honoured.
7. **Speed on small VRAM is a real cost, not a rounding error.** 8.4 minutes per image on an 8 GB
   card via CPU offload, against roughly a minute on a 16 GB T4. Fitting the model is not the same
   as running it usefully.
8. **Not FLUX.** SDXL was chosen so the notebook runs on a free T4 (see README). FLUX.2-klein is
   9B with a 24B text encoder and has no official ControlNet; FLUX.1-dev has a usable pose
   ControlNet but needs a paid runtime plus quantisation.

### Reproducibility

Fixed seeds reproduce a comparison on the same GPU and library versions. Exact pixels are not
portable across GPUs or `diffusers` versions, since cuDNN kernel selection and fp16 accumulation
order differ. The prompts, seeds and settings for every image above are in `prompts.md`.